In [ ]:
# arousalの変化が大きいLの発話の前後３発話を抽出して別フォルダに保存

In [ ]:
import pandas as pd

# 元の発話データ
original_csv = "/home/mitani/CSJ-emo-int_bunseki/0718/0720/LR_bunseki/data/vad_delta_L.csv"              
df_l = pd.read_csv(original_csv)
df_l.head()

,filename,emotion_izanami,intensity_pred_izanami,emotion_kushinada,intensity_pred_kushinada,valence,arousal,dominance,text,delta_valence,delta_arousal,delta_dominance,delta_int_kushinada
0,0001_L.wav,DIS,3.49,ACC,3.85,0.41,0.32,0.39,お願いします,NaN,NaN,NaN,NaN
1,0004_L.wav,ANT,2.11,JOY,2.29,0.52,0.39,0.40,今日はお疲れ様でした,0.11,0.07,0.01,1.56
2,0006_L.wav,ANT,2.27,ANT,0.76,0.30,0.36,0.40,どうでしたか,0.22,0.03,0.00,1.53
3,0007_L.wav,ACC,2.04,ACC,1.98,0.64,0.41,0.46,緊張します,0.34,0.05,0.06,1.22
4,0009_L.wav,FEA,1.88,SUR,2.34,0.66,0.65,0.57,<笑>,0.02,0.24,0.11,0.36


In [4]:
import pandas as pd
import os

# パーセンタイル95の発話データ
target_csv = "/home/mitani/CSJ-emo-int_bunseki/0718/0720/LR_bunseki/results/L_a/L_95_a.csv"  
df_target = pd.read_csv(target_csv)
df_target.head()

,filename,emotion_izanami,intensity_pred_izanami,emotion_kushinada,intensity_pred_kushinada,valence,arousal,dominance,text,delta_valence,delta_arousal,delta_dominance,delta_int_kushinada
0,0015_L.wav,SUR,1.17,SUR,2.85,0.88,0.77,0.74,<笑>,0.55,0.61,0.60,2.85
1,0034_L.wav,FEA,2.32,JOY,3.91,0.26,0.11,0.18,(F あー)本当本当,0.49,0.61,0.48,0.59
2,0047_L.wav,FEA,4.40,SUR,2.53,0.86,0.80,0.74,<笑>,0.50,0.60,0.53,1.87
3,0054_L.wav,ACC,1.19,SUR,5.58,0.96,0.91,0.85,<笑>,0.56,0.68,0.63,0.72
4,0098_L.wav,ANT,0.90,FEA,2.91,0.30,0.01,0.13,と思ったんですけど,0.01,0.56,0.41,0.39


In [ ]:
import pandas as pd
import os
import shutil

# 音声ファイル(1_L.wav形式)
wav_dir = "/home/mitani/CSJ-emo-int_bunseki/0718/0720/LR_bunseki/data/Segments/D03F0001/L"

# 保存先
output_root = "/home/mitani/CSJ-emo-int_bunseki/0718/0720/LR_bunseki/results/eval_q/L_95_a"
os.makedirs(output_root, exist_ok=True)

# 前後3発話
context = 3

# filename -> 行番号 の辞書
filename_to_idx = {
    filename: idx
    for idx, filename in enumerate(df_l["filename"])
}

# ==========================
# 抽出
# ==========================

for target_filename in df_target["filename"]:

    # 元データに存在するか確認
    if target_filename not in filename_to_idx:
        print(f"Not found in original csv: {target_filename}")
        continue

    idx = filename_to_idx[target_filename]

    print(f"{idx} -> {target_filename}")

    # 前後3発話
    start = max(0, idx - context)
    end = min(len(df_l), idx + context + 1)

    context_df = df_l.iloc[start:end]

    # 保存フォルダ
    folder = os.path.join(
        output_root,
        os.path.splitext(target_filename)[0]
    )
    os.makedirs(folder, exist_ok=True)

    # context.csv保存
    context_df.to_csv(
        os.path.join(folder, "context.csv"),
        index=False,
        encoding="utf-8-sig"
    )

    # target情報保存
    with open(os.path.join(folder, "target.txt"), "w", encoding="utf-8") as f:
        f.write(f"Original index : {idx}\n")
        f.write(f"Target filename : {target_filename}\n")
        f.write(f"Context range : {start} - {end - 1}\n")

    # 修正
    for _, row in context_df.iterrows():

        # CSVのファイル名
        csv_name = row["filename"]

        # 0001_L.wav → 1_L.wav
        num = int(csv_name.split("_")[0])     # 0001 → 1
        wav_name = f"{num}_L.wav"

        src = os.path.join(wav_dir, wav_name)

        if not os.path.exists(src):
            print(f"Not found: {src}")
            continue

        # 保存時はCSVの名前のままにする（見やすい）
        if csv_name == target_filename:
            dst_name = "TARGET_" + csv_name
        else:
            dst_name = csv_name

        dst = os.path.join(folder, dst_name)

        shutil.copy2(src, dst)

print("完了")

7 -> 0015_L.wav
18 -> 0034_L.wav
24 -> 0047_L.wav
29 -> 0054_L.wav
51 -> 0098_L.wav
57 -> 0116_L.wav
60 -> 0122_L.wav
68 -> 0135_L.wav
69 -> 0137_L.wav
104 -> 0211_L.wav
129 -> 0256_L.wav
132 -> 0263_L.wav
152 -> 0301_L.wav
170 -> 0334_L.wav
176 -> 0349_L.wav
179 -> 0357_L.wav
194 -> 0387_L.wav
203 -> 0410_L.wav
217 -> 0435_L.wav
224 -> 0447_L.wav
248 -> 0500_L.wav
251 -> 0504_L.wav
完了


In [8]:
# パーセンタイル90
import pandas as pd
import shutil
import os

# 変数の設定
q = 90          # quantile
f = "a"         # feature
sp = "L"        # speaker

# 元の発話データ
original_csv = f"/home/mitani/CSJ-emo-int_bunseki/0718/0720/LR_bunseki/data/vad_delta_{sp}.csv"              
df_l = pd.read_csv(original_csv)

# パーセンタイル90の発話データ
target_csv = f"/home/mitani/CSJ-emo-int_bunseki/0718/0720/LR_bunseki/results/{sp}_{f}/{sp}_{q}_{f}.csv"  
df_target = pd.read_csv(target_csv)


# 音声ファイル(1_L.wav形式)
wav_dir = f"/home/mitani/CSJ-emo-int_bunseki/0718/0720/LR_bunseki/data/Segments/D03F0001/{sp}"

# 保存先
output_root = f"/home/mitani/CSJ-emo-int_bunseki/0718/0720/LR_bunseki/results/eval_q/{sp}_{q}_{f}"
os.makedirs(output_root, exist_ok=True)

# 前後3発話
context = 3

# filename -> 行番号 の辞書
filename_to_idx = {
    filename: idx
    for idx, filename in enumerate(df_l["filename"])
}

# ==========================
# 抽出
# ==========================

for target_filename in df_target["filename"]:

    # 元データに存在するか確認
    if target_filename not in filename_to_idx:
        print(f"Not found in original csv: {target_filename}")
        continue

    idx = filename_to_idx[target_filename]

    print(f"{idx} -> {target_filename}")

    # 前後3発話
    start = max(0, idx - context)
    end = min(len(df_l), idx + context + 1)

    context_df = df_l.iloc[start:end]

    # 保存フォルダ
    folder = os.path.join(
        output_root,
        os.path.splitext(target_filename)[0]
    )
    os.makedirs(folder, exist_ok=True)

    # context.csv保存
    context_df.to_csv(
        os.path.join(folder, "context.csv"),
        index=False,
        encoding="utf-8-sig"
    )

    # target情報保存
    with open(os.path.join(folder, "target.txt"), "w", encoding="utf-8") as f:
        f.write(f"Original index : {idx}\n")
        f.write(f"Target filename : {target_filename}\n")
        f.write(f"Context range : {start} - {end - 1}\n")

    # 修正
    for _, row in context_df.iterrows():

        # CSVのファイル名
        csv_name = row["filename"]

        # 0001_L.wav → 1_L.wav
        num = int(csv_name.split("_")[0])     # 0001 → 1
        wav_name = f"{num}_L.wav"

        src = os.path.join(wav_dir, wav_name)

        if not os.path.exists(src):
            print(f"Not found: {src}")
            continue

        # 保存時はCSVの名前のままにする（見やすい）
        if csv_name == target_filename:
            dst_name = "TARGET_" + csv_name
        else:
            dst_name = csv_name

        dst = os.path.join(folder, dst_name)

        shutil.copy2(src, dst)

print("完了")

5 -> 0012_L.wav
7 -> 0015_L.wav
18 -> 0034_L.wav
24 -> 0047_L.wav
29 -> 0054_L.wav
51 -> 0098_L.wav
57 -> 0116_L.wav
60 -> 0122_L.wav
68 -> 0135_L.wav
69 -> 0137_L.wav
89 -> 0189_L.wav
104 -> 0211_L.wav
109 -> 0218_L.wav
129 -> 0256_L.wav
132 -> 0263_L.wav
152 -> 0301_L.wav
162 -> 0318_L.wav
170 -> 0334_L.wav
176 -> 0349_L.wav
179 -> 0357_L.wav
190 -> 0381_L.wav
194 -> 0387_L.wav
201 -> 0405_L.wav
203 -> 0410_L.wav
217 -> 0435_L.wav
221 -> 0442_L.wav
224 -> 0447_L.wav
248 -> 0500_L.wav
251 -> 0504_L.wav
252 -> 0505_L.wav
完了


In [9]:
# パーセンタイル85
import pandas as pd
import shutil
import os

# 変数の設定
q = 85
f = "a"         # feature
sp = "L"        # speaker

# 元の発話データ
original_csv = f"/home/mitani/CSJ-emo-int_bunseki/0718/0720/LR_bunseki/data/vad_delta_{sp}.csv"              
df_l = pd.read_csv(original_csv)

# パーセンタイル90の発話データ
target_csv = f"/home/mitani/CSJ-emo-int_bunseki/0718/0720/LR_bunseki/results/{sp}_{f}/{sp}_{q}_{f}.csv"  
df_target = pd.read_csv(target_csv)


# 音声ファイル(1_L.wav形式)
wav_dir = f"/home/mitani/CSJ-emo-int_bunseki/0718/0720/LR_bunseki/data/Segments/D03F0001/{sp}"

# 保存先
output_root = f"/home/mitani/CSJ-emo-int_bunseki/0718/0720/LR_bunseki/results/eval_q/{sp}_{q}_{f}"
os.makedirs(output_root, exist_ok=True)

# 前後3発話
context = 3

# filename -> 行番号 の辞書
filename_to_idx = {
    filename: idx
    for idx, filename in enumerate(df_l["filename"])
}

# ==========================
# 抽出
# ==========================

for target_filename in df_target["filename"]:

    # 元データに存在するか確認
    if target_filename not in filename_to_idx:
        print(f"Not found in original csv: {target_filename}")
        continue

    idx = filename_to_idx[target_filename]

    print(f"{idx} -> {target_filename}")

    # 前後3発話
    start = max(0, idx - context)
    end = min(len(df_l), idx + context + 1)

    context_df = df_l.iloc[start:end]

    # 保存フォルダ
    folder = os.path.join(
        output_root,
        os.path.splitext(target_filename)[0]
    )
    os.makedirs(folder, exist_ok=True)

    # context.csv保存
    context_df.to_csv(
        os.path.join(folder, "context.csv"),
        index=False,
        encoding="utf-8-sig"
    )

    # target情報保存
    with open(os.path.join(folder, "target.txt"), "w", encoding="utf-8") as f:
        f.write(f"Original index : {idx}\n")
        f.write(f"Target filename : {target_filename}\n")
        f.write(f"Context range : {start} - {end - 1}\n")

    # 修正
    for _, row in context_df.iterrows():

        # CSVのファイル名
        csv_name = row["filename"]

        # 0001_L.wav → 1_L.wav
        num = int(csv_name.split("_")[0])     # 0001 → 1
        wav_name = f"{num}_L.wav"

        src = os.path.join(wav_dir, wav_name)

        if not os.path.exists(src):
            print(f"Not found: {src}")
            continue

        # 保存時はCSVの名前のままにする（見やすい）
        if csv_name == target_filename:
            dst_name = "TARGET_" + csv_name
        else:
            dst_name = csv_name

        dst = os.path.join(folder, dst_name)

        shutil.copy2(src, dst)

print("完了")

5 -> 0012_L.wav
7 -> 0015_L.wav
10 -> 0020_L.wav
11 -> 0022_L.wav
16 -> 0030_L.wav
18 -> 0034_L.wav
22 -> 0042_L.wav
24 -> 0047_L.wav
29 -> 0054_L.wav
31 -> 0058_L.wav
37 -> 0072_L.wav
51 -> 0098_L.wav
57 -> 0116_L.wav
60 -> 0122_L.wav
68 -> 0135_L.wav
69 -> 0137_L.wav
82 -> 0169_L.wav
89 -> 0189_L.wav
104 -> 0211_L.wav
109 -> 0218_L.wav
118 -> 0234_L.wav
122 -> 0243_L.wav
129 -> 0256_L.wav
130 -> 0258_L.wav
131 -> 0262_L.wav
132 -> 0263_L.wav
152 -> 0301_L.wav
162 -> 0318_L.wav
170 -> 0334_L.wav
172 -> 0339_L.wav
176 -> 0349_L.wav
179 -> 0357_L.wav
185 -> 0370_L.wav
187 -> 0374_L.wav
188 -> 0377_L.wav
190 -> 0381_L.wav
193 -> 0385_L.wav
194 -> 0387_L.wav
195 -> 0389_L.wav
201 -> 0405_L.wav
202 -> 0409_L.wav
203 -> 0410_L.wav
217 -> 0435_L.wav
218 -> 0438_L.wav
221 -> 0442_L.wav
224 -> 0447_L.wav
240 -> 0482_L.wav
248 -> 0500_L.wav
251 -> 0504_L.wav
252 -> 0505_L.wav
完了


In [10]:
# パーセンタイル80
import pandas as pd
import shutil
import os

# 変数の設定
q = 80
f = "a"         # feature
sp = "L"        # speaker

# 元の発話データ
original_csv = f"/home/mitani/CSJ-emo-int_bunseki/0718/0720/LR_bunseki/data/vad_delta_{sp}.csv"              
df_l = pd.read_csv(original_csv)

# パーセンタイル90の発話データ
target_csv = f"/home/mitani/CSJ-emo-int_bunseki/0718/0720/LR_bunseki/results/{sp}_{f}/{sp}_{q}_{f}.csv"  
df_target = pd.read_csv(target_csv)


# 音声ファイル(1_L.wav形式)
wav_dir = f"/home/mitani/CSJ-emo-int_bunseki/0718/0720/LR_bunseki/data/Segments/D03F0001/{sp}"

# 保存先
output_root = f"/home/mitani/CSJ-emo-int_bunseki/0718/0720/LR_bunseki/results/eval_q/{sp}_{q}_{f}"
os.makedirs(output_root, exist_ok=True)

# 前後3発話
context = 3

# filename -> 行番号 の辞書
filename_to_idx = {
    filename: idx
    for idx, filename in enumerate(df_l["filename"])
}

# ==========================
# 抽出
# ==========================

for target_filename in df_target["filename"]:

    # 元データに存在するか確認
    if target_filename not in filename_to_idx:
        print(f"Not found in original csv: {target_filename}")
        continue

    idx = filename_to_idx[target_filename]

    print(f"{idx} -> {target_filename}")

    # 前後3発話
    start = max(0, idx - context)
    end = min(len(df_l), idx + context + 1)

    context_df = df_l.iloc[start:end]

    # 保存フォルダ
    folder = os.path.join(
        output_root,
        os.path.splitext(target_filename)[0]
    )
    os.makedirs(folder, exist_ok=True)

    # context.csv保存
    context_df.to_csv(
        os.path.join(folder, "context.csv"),
        index=False,
        encoding="utf-8-sig"
    )

    # target情報保存
    with open(os.path.join(folder, "target.txt"), "w", encoding="utf-8") as f:
        f.write(f"Original index : {idx}\n")
        f.write(f"Target filename : {target_filename}\n")
        f.write(f"Context range : {start} - {end - 1}\n")

    # 修正
    for _, row in context_df.iterrows():

        # CSVのファイル名
        csv_name = row["filename"]

        # 0001_L.wav → 1_L.wav
        num = int(csv_name.split("_")[0])     # 0001 → 1
        wav_name = f"{num}_L.wav"

        src = os.path.join(wav_dir, wav_name)

        if not os.path.exists(src):
            print(f"Not found: {src}")
            continue

        # 保存時はCSVの名前のままにする（見やすい）
        if csv_name == target_filename:
            dst_name = "TARGET_" + csv_name
        else:
            dst_name = csv_name

        dst = os.path.join(folder, dst_name)

        shutil.copy2(src, dst)

print("完了")

5 -> 0012_L.wav
7 -> 0015_L.wav
10 -> 0020_L.wav
11 -> 0022_L.wav
16 -> 0030_L.wav
18 -> 0034_L.wav
22 -> 0042_L.wav
24 -> 0047_L.wav
29 -> 0054_L.wav
31 -> 0058_L.wav
33 -> 0062_L.wav
37 -> 0072_L.wav
48 -> 0092_L.wav
51 -> 0098_L.wav
57 -> 0116_L.wav
58 -> 0119_L.wav
59 -> 0120_L.wav
60 -> 0122_L.wav
65 -> 0131_L.wav
67 -> 0134_L.wav
68 -> 0135_L.wav
69 -> 0137_L.wav
82 -> 0169_L.wav
89 -> 0189_L.wav
104 -> 0211_L.wav
109 -> 0218_L.wav
110 -> 0220_L.wav
111 -> 0223_L.wav
115 -> 0230_L.wav
118 -> 0234_L.wav
122 -> 0243_L.wav
126 -> 0251_L.wav
129 -> 0256_L.wav
130 -> 0258_L.wav
131 -> 0262_L.wav
132 -> 0263_L.wav
152 -> 0301_L.wav
162 -> 0318_L.wav
170 -> 0334_L.wav
171 -> 0337_L.wav
172 -> 0339_L.wav
175 -> 0345_L.wav
176 -> 0349_L.wav
179 -> 0357_L.wav
183 -> 0364_L.wav
185 -> 0370_L.wav
187 -> 0374_L.wav
188 -> 0377_L.wav
190 -> 0381_L.wav
193 -> 0385_L.wav
194 -> 0387_L.wav
195 -> 0389_L.wav
201 -> 0405_L.wav
202 -> 0409_L.wav
203 -> 0410_L.wav
217 -> 0435_L.wav
218 -> 0438_L.wav
